In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, when, to_date

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("DataLakeAssignment") \
    .getOrCreate()

bronze_path = "data_lake/bronze/sales.csv"

# 2. Load and Inspect with Spark
try:
    df = spark.read.csv(bronze_path, header=True, inferSchema=True)

    print("--- Original Schema ---")
    df.printSchema()

    print("\n--- Sample Rows ---")
    df.show(5)

    initial_count = df.count()
    print(f"\nTotal Initial Records: {initial_count}")

    # Identify Duplicates
    duplicate_count = initial_count - df.dropDuplicates().count()
    print(f"Exact Duplicate Rows Found: {duplicate_count}")

except Exception as e:
    print(f"Error loading dataset. Did you upload sales.csv to {bronze_path}? Error: {e}")

--- Original Schema ---
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)


--- Sample Rows ---
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+--------

In [4]:
# 3. Basic Preprocessing - Silver Layer

# 3.1 Remove exact duplicate rows
df_cleaned = df.dropDuplicates()

# 3.2 Handle missing values (Dropping rows that are entirely null)
df_cleaned = df_cleaned.dropna(how='all')

# 3.3 Trim spaces and standardize capitalization for text fields
string_cols = [c.name for c in df_cleaned.schema.fields if c.dataType.typeName() == 'string']
for c in string_cols:
    df_cleaned = df_cleaned.withColumn(c, trim(lower(col(c))))

# Helper for case-insensitive column matching
cols_lower = [c.lower() for c in df_cleaned.columns]

# 3.4 Validate numbers (Set negative quantities/prices to 0 or null as a reasonable approach)
if 'quantity' in cols_lower:
    actual_col = df_cleaned.columns[cols_lower.index('quantity')]
    df_cleaned = df_cleaned.withColumn(actual_col, when(col(actual_col) < 0, 0).otherwise(col(actual_col)))

if 'unit price' in cols_lower:
    actual_col = df_cleaned.columns[cols_lower.index('unit price')]
    df_cleaned = df_cleaned.withColumn(actual_col, when(col(actual_col) < 0, 0.0).otherwise(col(actual_col)))
elif 'unit_price' in cols_lower:
    actual_col = df_cleaned.columns[cols_lower.index('unit_price')]
    df_cleaned = df_cleaned.withColumn(actual_col, when(col(actual_col) < 0, 0.0).otherwise(col(actual_col)))

# 3.5 Convert valid dates (Attempt conversion to standard DateType)
if 'date' in cols_lower:
    actual_col = df_cleaned.columns[cols_lower.index('date')]
    df_cleaned = df_cleaned.withColumn(actual_col, to_date(col(actual_col)))

final_count = df_cleaned.count()

In [5]:
# Save the resulting DataFrame inside the Silver folder in CSV format
silver_path = "data_lake/silver/preprocessed_sales/"
df_cleaned.write.mode("overwrite").csv(silver_path, header=True)
print("Saved preprocessed data to Silver layer!")

Saved preprocessed data to Silver layer!


In [6]:
# 4. Verification
print(f"Row count before preprocessing: {initial_count}")
print(f"Row count after preprocessing:  {final_count}")

print("\n--- Reading Silver Data ---")
df_silver = spark.read.csv(silver_path, header=True, inferSchema=True)
df_silver.show(5)

print("\n--- Preprocessing Summary ---")
print("- Removed exact duplicate rows.")
print("- Handled missing values by dropping entirely null rows.")
print("- Trimmed whitespace and converted text fields to lowercase.")
print("- Validated numerical fields (Quantity/UnitPrice) and handled negatives.")
print("- Converted Date field to standard Spark DateType.")

Row count before preprocessing: 1000
Row count after preprocessing:  990

--- Reading Silver Data ---
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      c0005|      sakshi joshi|            football|   sports|       2|     670.0|              30|     credit card| bengaluru|  karnataka|2025-06-06|   cancelled|
|  100333|      c0263|siddharth kulkarni|    programming book|    books|       4|     670.0|               0|     net banking|      pune|maharashtra|2025-05-08|   delivered|
|  100506|      c0228|     d